# Lookback Length Analysis

This notebook analyzes model performance across different lookback lengths and generates visualizations showing how MSE changes as lookback length increases.

**Author**: MODE Team

**Date**: 2025-12-24

## 1. Import Libraries

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
from pathlib import Path
import matplotlib.ticker as ticker

# Models to plot — comment out any model you don't want to display
MODELS = [
    "ManiMamba",
    "interPDN",
    "BiMamba4TS",
    "S_Mamba",
    "iTransformer",
    "Informer",
    "Reformer",
    "Transformer",
]

# Color for each model curve
MODEL_COLORS = {
    "ManiMamba":      "#C50000",
    "interPDN":       "#0073EE",
    "BiMamba4TS":     "#FF7878",
    "S_Mamba":        "#8AC500",
    "iTransformer":   "#7A7A7A",
    "Informer":       "#00B8E1",
    "Reformer":       "#EFCB00",
    "Transformer":    "#B439C7",

}

MODEL_MARKERS = {
    "ManiMamba":      "o",
    "interPDN":       "s",
    "BiMamba4TS":     "^",
    "S_Mamba":        "v",
    "iTransformer":   "o",
    "Informer":       "s",
    "Reformer":       "^",
    "Transformer":    "v",
}

MARKER_SIZE = 3.5

DISPLAY_NAMES = {"ManiMamba": "SPDM"}

## 2. Define Data Parsing Functions

In [ ]:
def parse_result_file(filepath):
    """
    Parse a model result file and extract MSE and MAE values.
    
    Extracts: dataset, lookback, pred_len, model from each line
    """
    results = {}
    model_name = Path(filepath).stem

    with open(filepath, 'r') as f:
        lines = f.readlines()

    current_dataset = None
    current_lookback = None

    for i, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue

        # Check for dataset header: (Dataset ModelName)
        match = re.match(r'\((\w+)\s+(\w+)\)', line)
        if match:
            dataset = match.group(1)
            current_dataset = dataset
            if dataset not in results:
                results[dataset] = {'model_name': model_name, 'mse': {}, 'mae': {}}
            continue

        # Parse experiment line: Dataset_lookback_pred_model[_des:|...] or [|...]
        # Match pattern: {dataset}_{lookback}_{pred_len}_{model}[delimiter]
        if current_dataset and not line.startswith('mse:'):
            # Try to extract lookback from the line using regex
            # Pattern: {dataset}_{lookback}_{pred_len}_{model} followed by delimiter
            pattern = r'^' + re.escape(current_dataset) + r'_(\d+)_(\d+)_[^_\s|]+(?:[_|].*)?$'
            match = re.match(pattern, line)
            
            if match:
                current_lookback = int(match.group(1))
                continue

        # Parse metrics line
        if line.startswith('mse:') and current_dataset and current_lookback is not None:
            metrics = {}
            for metric in line.split(','):
                key, value = metric.strip().split(':')
                metrics[key] = float(value)

            if current_lookback not in results[current_dataset]['mse']:
                results[current_dataset]['mse'][current_lookback] = metrics['mse']
                results[current_dataset]['mae'][current_lookback] = metrics['mae']
            current_lookback = None

    return results

In [ ]:
def load_all_results(directory):
    """Load all model results from the directory, filtering out MODE models."""
    result_files = glob(os.path.join(directory, '*.txt'))
    all_results = {}

    for filepath in result_files:
        model_name = Path(filepath).stem

        # Filter out MODE
        if model_name == 'MODE':
            continue

        model_results = parse_result_file(filepath)
        for dataset, data in model_results.items():
            if dataset not in all_results:
                all_results[dataset] = {}
            all_results[dataset][model_name] = {
                'mse': data['mse'],
                'mae': data['mae']
            }

    return all_results

## 3. Load Data

In [ ]:
# Get the directory containing this notebook
notebook_dir = Path.cwd()

print("Loading model results...")
all_results = load_all_results(notebook_dir)

print(f"Found results for datasets: {list(all_results.keys())}")
print(f"Models: {list(all_results.get('ETTm1', {}).keys())}")

## 4. Create Summary Table

In [ ]:
def create_summary_table(all_results):
    """Create a summary table for all datasets and models."""
    summary_data = []

    for dataset in ['ETTm1', 'Weather', 'PEMS08']:
        if dataset not in all_results:
            continue

        for model_name in sorted(all_results[dataset].keys()):
            model_data = all_results[dataset][model_name]

            for lookback in sorted(model_data['mse'].keys()):
                summary_data.append({
                    'Dataset': dataset,
                    'Model': model_name,
                    'Lookback': lookback,
                    'MSE': model_data['mse'][lookback],
                    'MAE': model_data['mae'][lookback]
                })

    df = pd.DataFrame(summary_data)
    return df

summary_df = create_summary_table(all_results)

## 5. Create Visualization

In [ ]:
def plot_lookback_analysis(all_results, output_dir):
    """Generate lookback analysis plots for each dataset."""
    
    datasets = ['ETTm1', 'Weather', 'PEMS08']
    
    # Different lookback values for PEMS08
    lookback_config = {
        'ETTm1': [48, 96, 192, 336, 720],
        'Weather': [48, 96, 192, 336, 720],
        'PEMS08': [48, 96, 144, 192]
    }
    
    # Dashed style for multivariate variants
    _DASH_MODELS = {'Transformer_M', 'Informer_M', 'Reformer_M'}

    fig, axes = plt.subplots(1, 3, figsize=(8, 2.3), dpi=600)

    legend_items = {}

    for idx, dataset in enumerate(datasets):
        if dataset not in all_results:
            axes[idx].text(0.5, 0.5, 'No Data', ha='center', va='center', transform=axes[idx].transAxes, fontsize=5, color='#000000')
            axes[idx].set_title(dataset, fontsize=6, color='#000000')
            continue

        ax = axes[idx]
        lookback_values = lookback_config[dataset]
        lookback_indices = np.arange(len(lookback_values))

        # Linear scale with evenly spaced ticks
        ax.set_yscale('linear')
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5, prune=None))
        formatter = ticker.FormatStrFormatter('%.2f')
        ax.yaxis.set_major_formatter(formatter)

        _ordered = [m for m in MODELS if m in all_results[dataset]]
        if 'ManiMamba' in _ordered:
            _ordered.remove('ManiMamba')
            _ordered.append('ManiMamba')
        for model_name in _ordered:
            color = MODEL_COLORS.get(model_name, '#000000')
            style = '--' if model_name in _DASH_MODELS else '-'

            model_data = all_results[dataset][model_name]
            mse_values = [model_data['mse'].get(lb, np.nan) for lb in lookback_values]

            if not all(np.isnan(mse_values)):
                line, = ax.plot(
                    lookback_indices,
                    mse_values,
                    marker=MODEL_MARKERS.get(model_name, 'o'),
                    linewidth=0.8,
                    markersize=MARKER_SIZE,
                    color=color,
                    linestyle=style,
                    label=DISPLAY_NAMES.get(model_name, model_name),
                    alpha=0.95
                )
                if model_name not in legend_items:
                    legend_items[model_name] = line

        ax.set_title(f'{dataset}', fontsize=7, fontweight='bold', color='#000000')
        ax.set_xlabel('Lookback Length', fontsize=6, color="black", fontweight="bold")
        if idx == 0:
            ax.set_ylabel('MSE', fontsize=7, color="black", fontweight="bold")

        for spine in ax.spines.values():
            spine.set_color('#000000')
            spine.set_linewidth(0.6)

        # Harmonize tick label sizes/colors for both axes
        ax.tick_params(axis='both', which='both', colors='#000000', labelsize=5, width=0.6)

        ax.grid(False)
        ax.xaxis.grid(True, color='#d0d0d0', linestyle='-', alpha=0.35)

        ax.set_xticks(lookback_indices)
        ax.set_xticklabels(lookback_values, fontsize=5, color='#000000')

        if dataset == 'PEMS08':
            ax.set_ylim(top=0.7)

        if dataset == 'ETTm1':
            ax.set_ylim(top=0.7)

        if dataset == 'Weather':
            ax.set_ylim(top=0.5)

    # Sort legend by MODELS list order
    sorted_models = [m for m in MODELS if m in legend_items]

    handles = [legend_items[m] for m in sorted_models]

    # Force single row for legend
    ncol = len(sorted_models)

    # Reduce spacing between legend and plots
    _display = [DISPLAY_NAMES.get(m, m) for m in sorted_models]
    leg = fig.legend(
        handles,
        _display,
        loc='lower center',
        bbox_to_anchor=(0.5, 0.02),
        ncol=ncol,
        fontsize=6,
        frameon=False,
        columnspacing=1.0,
        handlelength=2.0,
    )

    # Bold 'SPDM' in legend
    for text in leg.get_texts():
        if text.get_text() == 'SPDM':
            text.set_weight('bold')

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.28, wspace=0.15)

    return fig

# Generate the plot
fig = plot_lookback_analysis(all_results, notebook_dir)